## INICIALIZAÇÃO

In [1]:
import os
import sys
import datetime
import pandas as pd

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

import random
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from cnnbilstm import CNNBiLSTM
from resnetlstm import ResNetLSTM
from loaders import build_mel_dataloader, build_video_dataloader
from balanced_dataset import balanced_df

In [2]:
print("Versão do Torch:", torch.__version__)
print("Versão do Torchvision:", torchvision.__version__)
print(f"CUDA disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Versão do CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

seed = 435
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Semente: {seed}")
print(f"Dispositivo: {device}")

Versão do Torch: 2.5.1+cu121
Versão do Torchvision: 0.20.1+cu121
CUDA disponível: True
Versão do CUDA: 12.1
GPU: NVIDIA RTX A4500
Semente: 435
Dispositivo: cuda


In [3]:
CSV_PATH = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"

train_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

valid_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

train_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 36304/37620 exemplos válidos.
Dataset: 11224/12540 exemplos válidos.
Dataset: 37620/37620 exemplos válidos.
Dataset: 12540/12540 exemplos válidos.


## AVALIAÇÃO

#### Calcular o CCC

Para validar se a função gerada pela rede é boa, a forma ideal de comparar com o seu Ground Truth (que também deve ser mapeado para o tempo) é usando o CCC, que mede a concordância de trajetórias temporais:

$$ρ_c​= \dfrac{2ρσ_x​σ_y}{σ_x^2​+σ_y^2+(μ_x​−μ_y​)^2}​​$$

Onde ρ é a correlação de Pearson, μ é a média e σ2 é a variância. O CCC pune o modelo não apenas se a curva subir e descer nos momentos errados, mas também se houver um deslocamento de escala (ex: o modelo captou a variação, mas os valores absolutos estão muito baixos).

In [4]:
def calculate_ccc(y_true, y_pred):
    """Calcula o Concordance Correlation Coefficient (CCC)."""
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)

    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    numerator = 2 * covariance
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2

    return 0.0 if denominator == 0 else numerator / denominator

# TREINAMENTO

In [5]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    epochs=10,
    device="cuda",
    patience=5,
    checkpoint_path="best_model.pth",
    nome_modelo='ESQUECEU_O_NOME'
):
    """
    Treina a rede e a avalia a cada época usando o CCC como métrica principal.

    Argumentos:
        model:            Modelo PyTorch.
        train_loader:     DataLoader para os dados de treino.
        val_loader:       DataLoader para os dados de validação.
        optimizer:        Otimizador (ex: AdamW).
        criterion:        Função de perda (ex: nn.MSELoss()).
        scheduler:        Scheduler de learning rate (ex: ReduceLROnPlateau).
        epochs:           Número máximo de épocas.
        device:           Dispositivo de treino ("cuda" ou "cpu").
        patience:         Épocas sem melhora no CCC antes de acionar early stopping.
        checkpoint_path:  Caminho para salvar o melhor modelo.
    """

    assert nome_modelo != 'ESQUECEU_O_NOME', "Coloque um nome nesse modelo, por favor"

    model.to(device)
    best_val_ccc = -1.0 # CCC varia de - 1 a 1, quanto mais perto de 1, melhor
    epochs_no_improve = 0

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join("runs", f"arousal_{current_time}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):

        # ------------------------------------------------------------------ #
        # TREINO
        # ------------------------------------------------------------------ #
        model.train()
        train_loss = 0.0

        for videos, masks, targets in tqdm(
            train_loader, desc=f"Época {epoch+1}/{epochs} [treino]"
        ):
            videos = videos.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos).squeeze(1)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * videos.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------------------------ #
        # VALIDAÇÃO
        # ------------------------------------------------------------------ #
        model.eval()
        val_loss = 0.0
        all_true = []
        all_pred = []

        with torch.no_grad():
            for videos, masks, targets in tqdm(
                val_loader, desc=f"Época {epoch+1}/{epochs} [val]"
            ):
                videos = videos.to(device)
                targets = targets.to(device)

                outputs = model(videos).squeeze(1)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * videos.size(0)

                all_true.extend(targets.cpu().numpy())
                all_pred.extend(outputs.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(all_true) - np.array(all_pred)))
        val_ccc = calculate_ccc(np.array(all_true), np.array(all_pred))

        scheduler.step(val_loss)

        # ------------------------------------------------------------------ #
        # TENSORBOARD
        # ------------------------------------------------------------------ #
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE", val_mae, epoch)
        writer.add_scalar("Metrics/CCC", val_ccc, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Época [{epoch+1}/{epochs}]"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Val MAE: {val_mae:.4f}"
            f" | Val CCC: {val_ccc:.4f}"
            f" | LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ------------------------------------------------------------------ #
        # CHECKPOINT + EARLY STOPPING
        # ------------------------------------------------------------------ #
        if val_ccc > best_val_ccc:
            best_val_ccc = val_ccc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_ccc": best_val_ccc,
            }, checkpoint_path)
            print(f"Novo melhor modelo salvo! (CCC: {best_val_ccc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"Sem melhora por {epochs_no_improve}/{patience} épocas.")
            if epochs_no_improve >= patience:
                print("Early stopping ativado.")
                break

    writer.close()
    print(f"\nTreinamento concluído. Melhor CCC: {best_val_ccc:.4f}")

# TREINAMENTO C/ FINE-TUNING

In [6]:
def train_model_fine_tuning(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    epochs=10,
    device="cuda",
    patience=5,
    checkpoint_path="best_model.pth",
    unfreeze_epoch=3,
    nome_modelo='ESQUECEU_O_NOME'
):
    """
    Treina a rede e a avalia a cada época usando o CCC como métrica principal.
    Congela a ResNet inicial até atingir a 'unfreeze_epoch'.
    """

    assert nome_modelo != 'ESQUECEU_O_NOME', "Coloque um nome nesse modelo, por favor"

    model.to(device)
    best_val_ccc = -1.0 
    epochs_no_improve = 0

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join("runs", f"arousal_{nome_modelo}_{current_time}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    # ================================================================== #
    # CONFIGURAÇÃO INICIAL: CONGELAR A RESNET
    # ================================================================== #
    # Congela toda a ResNet, EXCETO a primeira camada de 1 canal que você alterou
    print("=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...")
    for name, param in model.cnn.named_parameters():
        if "0.weight" not in name:  # Mantém a primeira camada aberta pois ela foi reinicializada do zero
            param.requires_grad = False
            
    # Força o otimizador a olhar apenas para os parâmetros que estão ativos no momento
    # (Evita que o otimizador gaste memória/etapas com gradientes congelados)
    optimizer.param_groups[0]['params'] = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(epochs):

        # ================================================================== #
        # LÓGICA DE DESCONGELAMENTO (UNFREEZE)
        # ================================================================== #
        if epoch == unfreeze_epoch:
            print(f"\n[ÉPOCA {epoch+1}] => ATENÇÃO: Descongelando os pesos da ResNet para Fine-Tuning completo!")
            
            # Ativa o gradiente para absolutamente todos os parâmetros do modelo
            for param in model.parameters():
                param.requires_grad = True
                
            # CRUCIAL: Atualiza o otimizador para passar a incluir os novos pesos da ResNet
            # Mantém a Learning Rate atual que estava no param_groups
            current_lr = optimizer.param_groups[0]['lr']
            optimizer.param_groups[0]['params'] = list(model.parameters())
            
            # Opcional: Se quiser treinar a ResNet com uma LR menor para não destruir os pesos dela,
            # você poderia separar os grupos aqui. Mas para simplificar, mantivemos todos juntos.

        # ------------------------------------------------------------------ #
        # TREINO
        # ------------------------------------------------------------------ #
        model.train()
        train_loss = 0.0

        for videos, masks, targets in tqdm(
            train_loader, desc=f"Época {epoch+1}/{epochs} [treino]"
        ):
            videos = videos.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos).squeeze(1)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * videos.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------------------------ #
        # VALIDAÇÃO
        # ------------------------------------------------------------------ #
        model.eval()
        val_loss = 0.0
        all_true = []
        all_pred = []

        with torch.no_grad():
            for videos, masks, targets in tqdm(
                val_loader, desc=f"Época {epoch+1}/{epochs} [val]"
            ):
                videos = videos.to(device)
                targets = targets.to(device)

                outputs = model(videos).squeeze(1)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * videos.size(0)

                all_true.extend(targets.cpu().numpy())
                all_pred.extend(outputs.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(all_true) - np.array(all_pred)))
        val_ccc = calculate_ccc(np.array(all_true), np.array(all_pred))

        scheduler.step(val_loss)

        # ------------------------------------------------------------------ #
        # TENSORBOARD
        # ------------------------------------------------------------------ #
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE", val_mae, epoch)
        writer.add_scalar("Metrics/CCC", val_ccc, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        # Print explicativo do estado atual do treino
        status_resnet = "Congelada" if epoch < unfreeze_epoch else "Aberta (Fine-Tuning)"
        print(
            f"Época [{epoch+1}/{epochs}] [{status_resnet}]"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Val MAE: {val_mae:.4f}"
            f" | Val CCC: {val_ccc:.4f}"
            f" | LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ------------------------------------------------------------------ #
        # CHECKPOINT + EARLY STOPPING
        # ------------------------------------------------------------------ #
        if val_ccc > best_val_ccc:
            best_val_ccc = val_ccc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_ccc": best_val_ccc,
            }, checkpoint_path)
            print(f"Novo melhor modelo salvo! (CCC: {best_val_ccc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"Sem melhora por {epochs_no_improve}/{patience} épocas.")
            if epochs_no_improve >= patience:
                print("Early stopping ativado.")
                break

    writer.close()
    print(f"\nTreinamento concluído. Melhor CCC: {best_val_ccc:.4f}")

    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
    patience=5,
    epochs=15,

## Treino com dados balanceados

Com a finalidade de avaliar o treinamento do modelo com um dataset balanceado (50% de clips highlight e 50% não-highlights), vamos filtrar o conjunto de treino com base na quantidade de clips highlight em uma partida e adicionando apenas a quantidade correspondente de clips não-highlights.

Inicialmente o threshold para o arousal_score para que o clip ser=ja considerado highlight é 0.5. 

In [7]:
# balanceando os dados

all_data = pd.read_csv("/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv")

all_train = all_data[all_data['split'] == 'train'] # todos os dados do conjunto de treino
all_valid = all_data[all_data['split'] == 'valid']
all_test = all_data[all_data['split'] == 'test']

balanced_train = balanced_df(all_train, 'game_id', threshold=0.5, random_state=seed)
balanced_valid = balanced_df(all_valid, 'game_id', threshold=0.5, random_state=seed)
balanced_test = balanced_df(all_test, 'game_id', threshold=0.5, random_state=seed)

# salvar em csv
balanced_train.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_train.csv', index=False)
balanced_valid.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_val.csv', index=False)
balanced_test.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_test.csv', index=False)

In [8]:
# criação dos loaders:
TRAIN_PATH = '/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_train.csv'
VAL_PATH = '/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_val.csv'

train_mel_loader_balanced = build_mel_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

valid_mel_loader_balanced = build_mel_dataloader(
    csv_path=VAL_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

train_video_loader_balanced = build_video_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=0,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader_balanced = build_video_dataloader(
    csv_path=VAL_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=0,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 1467/1514 exemplos válidos.
Dataset: 579/654 exemplos válidos.
Dataset: 1514/1514 exemplos válidos.
Dataset: 654/654 exemplos válidos.


O volume do dataset diminui consideravelmente.

Antes do balanceamento:
- Dataset train mel: 36304/37620 exemplos válidos.
- Dataset val mel: 11224/12540 exemplos válidos.
- Dataset train video: 37620/37620 exemplos válidos.
- Dataset val video: 12540/12540 exemplos válidos.

Depois do balanceamento:
- Dataset train mel: 1467/1514 exemplos válidos.
- Dataset val mel: 579/654 exemplos válidos.
- Dataset train video: 1514/1514 exemplos válidos.
- Dataset val video: 654/654 exemplos válidos.

### Treino da CNNiLSTM

### Treino da ResNetLSTM com dados balanceados

### Treino da ResNetLSTM com dados balanceados COM fine-tuning

In [9]:
# ================================================================== #
# MODELO 1 — Baseline leve (menos parâmetros, menos risco de overfit)
# ================================================================== #
resnet_01 = ResNetLSTM(
    frame_step=5, hidden_size=128, num_layers=1,
    use_dropout=True, dropout_p=0.3,
).to(device)

optimizer_01 = AdamW(resnet_01.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_01 = ReduceLROnPlateau(optimizer_01, mode="min", patience=3, factor=0.5)

train_model_fine_tuning(
    model=resnet_01,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader,          # sem balanceamento na val
    optimizer=optimizer_01, criterion=nn.MSELoss(),
    scheduler=scheduler_01,
    epochs=100, device=device, patience=10,
    unfreeze_epoch=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_01_leve.pth",
    nome_modelo='RESNET_01_LEVE'
)

TensorBoard: runs/arousal_RESNET_01_LEVE_20260609-174927
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/100 [val]: 100%|██████████| 3135/3135 [24:42<00:00,  2.12it/s]


Época [1/100] [Congelada] | Train Loss: 0.2017 | Val Loss: 0.1891 | Val MAE: 0.4162 | Val CCC: 0.0236 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0236)


Época 2/100 [val]: 100%|██████████| 3135/3135 [24:35<00:00,  2.12it/s]


Época [2/100] [Congelada] | Train Loss: 0.1864 | Val Loss: 0.2450 | Val MAE: 0.4746 | Val CCC: 0.0183 | LR: 1.00e-04
Sem melhora por 1/10 épocas.


...

Época [13/100] [Aberta (Fine-Tuning)] | Train Loss: 0.0392 | Val Loss: 0.1105 | Val MAE: 0.2230 | Val CCC: 0.0706 | LR: 2.50e-05
Sem melhora por 8/10 épocas.


Época 14/100 [val]: 100%|██████████| 3135/3135 [24:53<00:00,  2.10it/s]


Época [14/100] [Aberta (Fine-Tuning)] | Train Loss: 0.0243 | Val Loss: 0.1376 | Val MAE: 0.2653 | Val CCC: 0.0599 | LR: 2.50e-05
Sem melhora por 9/10 épocas.


Época 15/100 [val]: 100%|██████████| 3135/3135 [24:49<00:00,  2.11it/s]

Época [15/100] [Aberta (Fine-Tuning)] | Train Loss: 0.0155 | Val Loss: 0.1761 | Val MAE: 0.3196 | Val CCC: 0.0500 | LR: 2.50e-05
Sem melhora por 10/10 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.0711


In [10]:
# ================================================================== #
# MODELO 2 — CCC Loss em vez de MSE
# ================================================================== #
def ccc_loss(pred, target):
    pred_mean   = pred.mean()
    target_mean = target.mean()
    covariance  = ((pred - pred_mean) * (target - target_mean)).mean()
    pred_var    = pred.var()
    target_var  = target.var()
    ccc = (2 * covariance) / (pred_var + target_var + (pred_mean - target_mean) ** 2 + 1e-8)
    return 1 - ccc

resnet_02 = ResNetLSTM(
    frame_step=5, hidden_size=256, num_layers=2,
    use_dropout=True, dropout_p=0.3,
).to(device)

optimizer_02 = AdamW(resnet_02.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_02 = ReduceLROnPlateau(optimizer_02, mode="min", patience=3, factor=0.5)

train_model_fine_tuning(
    model=resnet_02,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader,
    optimizer=optimizer_02, criterion=ccc_loss,
    scheduler=scheduler_02,
    epochs=100, device=device, patience=10,
    unfreeze_epoch=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_02_cccloss.pth",
    nome_modelo='RESNET_02_CCCLOSS'
)

TensorBoard: runs/arousal_RESNET_02_CCCLOSS_20260610-020027
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/100 [val]: 100%|██████████| 3135/3135 [24:48<00:00,  2.11it/s]


Época [1/100] [Congelada] | Train Loss: 0.8579 | Val Loss: 0.9958 | Val MAE: 0.2104 | Val CCC: 0.1118 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1118)


Época 2/100 [val]: 100%|██████████| 3135/3135 [24:48<00:00,  2.11it/s]


Época [2/100] [Congelada] | Train Loss: 0.7844 | Val Loss: 0.9952 | Val MAE: 0.3343 | Val CCC: 0.0824 | LR: 1.00e-04
Sem melhora por 1/10 épocas.


...

Época [9/100] [Aberta (Fine-Tuning)] | Train Loss: 0.7447 | Val Loss: 0.9936 | Val MAE: 0.2819 | Val CCC: 0.0737 | LR: 5.00e-05
Sem melhora por 8/10 épocas.


Época 10/100 [val]: 100%|██████████| 3135/3135 [24:51<00:00,  2.10it/s]


Época [10/100] [Aberta (Fine-Tuning)] | Train Loss: 0.7034 | Val Loss: 0.9904 | Val MAE: 0.3071 | Val CCC: 0.0683 | LR: 5.00e-05
Sem melhora por 9/10 épocas.


Época 11/100 [val]: 100%|██████████| 3135/3135 [24:54<00:00,  2.10it/s]

Época [11/100] [Aberta (Fine-Tuning)] | Train Loss: 0.6425 | Val Loss: 0.9911 | Val MAE: 0.2677 | Val CCC: 0.0879 | LR: 5.00e-05
Sem melhora por 10/10 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.1118


In [11]:
# ================================================================== #
# MODELO 3 — MSE + CCC Loss combinados (alpha=0.5)
# ================================================================== #
def combined_loss(pred, target, alpha=0.5):
    return nn.MSELoss()(pred, target) + alpha * ccc_loss(pred, target)

resnet_03 = ResNetLSTM(
    frame_step=5, hidden_size=256, num_layers=2,
    use_dropout=True, dropout_p=0.3,
).to(device)

optimizer_03 = AdamW(resnet_03.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_03 = ReduceLROnPlateau(optimizer_03, mode="min", patience=3, factor=0.5)

train_model_fine_tuning(
    model=resnet_03,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader,
    optimizer=optimizer_03, criterion=combined_loss,
    scheduler=scheduler_03,
    epochs=100, device=device, patience=10,
    unfreeze_epoch=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_03_combined.pth",
    nome_modelo='RESNET_03_COMBINED'
)

TensorBoard: runs/arousal_RESNET_03_COMBINED_20260610-080145
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/100 [val]: 100%|██████████| 3135/3135 [24:54<00:00,  2.10it/s]


Época [1/100] [Congelada] | Train Loss: 0.6681 | Val Loss: 0.6733 | Val MAE: 0.3812 | Val CCC: 0.0361 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0361)


Época 2/100 [val]: 100%|██████████| 3135/3135 [24:59<00:00,  2.09it/s]


Época [2/100] [Congelada] | Train Loss: 0.5956 | Val Loss: 0.7999 | Val MAE: 0.4555 | Val CCC: 0.0365 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0365)


...

Época [17/100] [Aberta (Fine-Tuning)] | Train Loss: 0.2084 | Val Loss: 0.6625 | Val MAE: 0.2741 | Val CCC: 0.0486 | LR: 2.50e-05
Sem melhora por 8/10 épocas.


Época 18/100 [val]: 100%|██████████| 3135/3135 [24:53<00:00,  2.10it/s]


Época [18/100] [Aberta (Fine-Tuning)] | Train Loss: 0.1923 | Val Loss: 0.6559 | Val MAE: 0.2682 | Val CCC: 0.0599 | LR: 2.50e-05
Sem melhora por 9/10 épocas.


Época 19/100 [val]: 100%|██████████| 3135/3135 [24:54<00:00,  2.10it/s]

Época [19/100] [Aberta (Fine-Tuning)] | Train Loss: 0.1866 | Val Loss: 0.6607 | Val MAE: 0.2793 | Val CCC: 0.0567 | LR: 2.50e-05
Sem melhora por 10/10 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.1039


In [ ]:
# ================================================================== #
# MODELO 4 — Unfreeze mais cedo + LR diferenciada por grupo
# ================================================================== #
resnet_04 = ResNetLSTM(
    frame_step=5, hidden_size=256, num_layers=2,
    use_dropout=True, dropout_p=0.3,
).to(device)

# LR menor para a ResNet, maior para o LSTM
optimizer_04 = AdamW([
    {"params": resnet_04.cnn.parameters(),  "lr": 1e-5},
    {"params": resnet_04.lstm.parameters(), "lr": 1e-4},
    {"params": resnet_04.head.parameters(),   "lr": 1e-4},
], weight_decay=1e-4)
scheduler_04 = ReduceLROnPlateau(optimizer_04, mode="min", patience=3, factor=0.5)

train_model_fine_tuning(
    model=resnet_04,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader,
    optimizer=optimizer_04, criterion=combined_loss,
    scheduler=scheduler_04,
    epochs=100, device=device, patience=10,
    unfreeze_epoch=3,               # descongela mais cedo pois a LR da ResNet já é pequena
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_04_lrdiff.pth",
    nome_modelo='RESNET_04_LRDIFF'
)

AttributeError: 'ResNetLSTM' object has no attribute 'fc'